# 01. 画像・動画生成サーバー (ComfyUI + GGUF)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hiirocreate/personalizeAI/blob/main/notebooks/01_comfyui_server.ipynb)

無料の Colab T4 GPU で ComfyUI を起動し、GGUF 量子化モデルで軽量に生成します。
- **画像**: Flux.1 schnell GGUF (4ステップ高速) / SDXL (イラスト: Animagine XL 4.0, 実写: RealVisXL 5.0)
- **動画**: Wan2.2 TI2V 5B GGUF (テキスト→動画 / 画像→動画)
- **LoRA**: `02_lora_training` で作った LoRA を Google Drive から自動読込

手順: ランタイム → ランタイムのタイプを変更 → **T4 GPU** → 上から順に ▶ 実行。
起動後に表示される URL を [Web UI](https://hiirocreate.github.io/personalizeAI/) に貼ると、ブラウザ/スマホから操作できます。

In [ ]:
#@title ① 設定
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown ダウンロードするモデル (T4 のディスク/メモリ節約のため必要なものだけ)
FLUX = True  #@param {type:"boolean"}
FLUX_QUANT = "Q4_K_S"  #@param ["Q3_K_S", "Q4_K_S", "Q5_K_S", "Q6_K", "Q8_0"]
SDXL_ILLUST = True  #@param {type:"boolean"}
SDXL_PHOTO = False  #@param {type:"boolean"}
WAN_VIDEO = True  #@param {type:"boolean"}
WAN_QUANT = "Q4_K_M"  #@param ["Q3_K_M", "Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0"]
#@markdown 追加モデル (Civitai 等の直リンク) `URL|models/サブフォルダ` をカンマ区切り
EXTRA = ""  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}

In [ ]:
#@title ② インストール & モデル取得 (初回 5〜10 分)

import os, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(cmd)
    return r.stdout

def dl(url, dst_dir, name=None):
    """aria2c で高速ダウンロード (既にあればスキップ)"""
    os.makedirs(dst_dir, exist_ok=True)
    name = name or url.split("/")[-1].split("?")[0]
    if os.path.exists(os.path.join(dst_dir, name)): return
    hdr = f'--header="Authorization: Bearer {HF_TOKEN}"' if HF_TOKEN and "huggingface.co" in url else ""
    print("↓", name); sh(f'aria2c -q -x16 -s16 -k1M --console-log-level=error {hdr} -d "{dst_dir}" -o "{name}" "{url}"')

if not os.path.exists("/usr/bin/aria2c"): sh("apt-get -qq install -y aria2")
if not os.path.exists("/content/personalizeAI"): sh("git clone -q --depth 1 https://github.com/hiirocreate/personalizeAI /content/personalizeAI")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/personalizeAI"
else:
    BASE = "/content/personalizeAI_data"
for d in ["loras", "outputs", "datasets", "3d"]: os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("データ保存先:", BASE)

C = "/content/ComfyUI"
if not os.path.exists(C):
    sh(f"git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI {C}")
    sh(f"git clone -q --depth 1 https://github.com/city96/ComfyUI-GGUF {C}/custom_nodes/ComfyUI-GGUF")
    sh(f"pip install -q -r {C}/requirements.txt gguf")
# LoRA と出力は Drive に直結
for sub, target in [("models/loras", "loras"), ("output", "outputs")]:
    p = f"{C}/{sub}"
    if not os.path.islink(p):
        sh(f'rm -rf "{p}" && ln -s "{BASE}/{target}" "{p}"')
M = f"{C}/models"
if FLUX:
    dl(f"https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-{FLUX_QUANT}.gguf", f"{M}/unet")
    dl(f"https://huggingface.co/city96/t5-v1_1-xxl-encoder-gguf/resolve/main/t5-v1_1-xxl-encoder-Q5_K_M.gguf", f"{M}/clip")
    dl(f"https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors", f"{M}/clip")
    dl(f"https://huggingface.co/Comfy-Org/Lumina_Image_2.0_Repackaged/resolve/main/split_files/vae/ae.safetensors", f"{M}/vae")
if SDXL_ILLUST:
    dl(f"https://huggingface.co/cagliostrolab/animagine-xl-4.0/resolve/main/animagine-xl-4.0-opt.safetensors", f"{M}/checkpoints")
if SDXL_PHOTO:
    dl(f"https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors", f"{M}/checkpoints")
if WAN_VIDEO:
    dl(f"https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/Wan2.2-TI2V-5B-{WAN_QUANT}.gguf", f"{M}/unet")
    dl(f"https://huggingface.co/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q5_K_M.gguf", f"{M}/clip")
    dl(f"https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors", f"{M}/vae")
for item in filter(None, [s.strip() for s in EXTRA.split(",")]):
    url, sub = item.split("|")
    dl(url, f"{C}/{sub}", None if "civitai" not in url else url.split("/")[-1].split("?")[0] + ".safetensors")
print("✅ 準備完了")

In [ ]:
#@title ③ ComfyUI 起動 + 外部公開URL発行 (Cloudflare Tunnel・無料/登録不要)
import re, time, socket, urllib.parse
from IPython.display import display, HTML
if not os.path.exists("/content/cloudflared"):
    sh("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared")
subprocess.Popen(f'cd {C} && python main.py --listen 127.0.0.1 --port 8188 --enable-cors-header "*" --preview-method auto > /content/comfy.log 2>&1', shell=True)
while socket.socket().connect_ex(("127.0.0.1", 8188)): time.sleep(1)
subprocess.Popen("/content/cloudflared tunnel --url http://127.0.0.1:8188 > /content/tunnel.log 2>&1", shell=True)
API = None
while not API:
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    API = m and m.group(0)
ui = "https://hiirocreate.github.io/personalizeAI/?api=" + urllib.parse.quote(API)
display(HTML(f'<h3>ComfyUI: <a href="{API}" target="_blank">{API}</a></h3>'
             f'<h3>Web UI (スマホ可): <a href="{ui}" target="_blank">{ui}</a></h3>'))

In [ ]:
#@title ④ (任意) ノートブック内で生成
import sys; sys.path.insert(0, "/content/personalizeAI")
from pai.comfy import Comfy
from IPython.display import Image, Video, display
MODE = "flux_gguf"  #@param ["flux_gguf", "sdxl", "wan22_video"]
PROMPT = "1girl, silver hair, school uniform, cherry blossoms, masterpiece, best quality"  #@param {type:"string"}
CKPT = "animagine-xl-4.0-opt.safetensors"  #@param ["animagine-xl-4.0-opt.safetensors", "RealVisXL_V5.0_fp16.safetensors"] {allow-input: true}
LORA = ""  #@param {type:"string"}
LORA_STRENGTH = 0.8  #@param {type:"number"}
WIDTH = 832  #@param {type:"integer"}
HEIGHT = 1216  #@param {type:"integer"}
SEED = -1  #@param {type:"integer"}
#@markdown 動画用: 開始画像パス (空ならテキスト→動画), フレーム数 (4n+1)
IMAGE = ""  #@param {type:"string"}
FRAMES = 49  #@param {type:"integer"}
p = dict(prompt=PROMPT, width=WIDTH, height=HEIGHT, seed=SEED, lora=LORA, lora_strength=LORA_STRENGTH)
if MODE == "flux_gguf": p["unet"] = f"flux1-schnell-{FLUX_QUANT}.gguf"
if MODE == "sdxl": p["ckpt"] = CKPT
if MODE == "wan22_video": p.update(unet=f"Wan2.2-TI2V-5B-{WAN_QUANT}.gguf", image=IMAGE, frames=FRAMES)
for f in Comfy().run(MODE, out_dir=f"{BASE}/outputs", **p):
    print(f); display(Video(f, embed=True, width=512) if f.endswith(".mp4") else Image(f, width=512))